<a href="https://colab.research.google.com/github/korkutanapa/DCASE2025_TASK2/blob/main/ORJ_DCASE_TDA_FEATURE_CREATOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!rm -rf /content/test/*

In [ ]:
# @title
# ============================================================
# INSTALL LIBRARIES
# ============================================================
!pip install -q librosa gudhi openpyxl tqdm

# ============================================================
# INSTALL / IMPORT
# ============================================================
!pip install -q openpyxl scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 33.1 MB/s eta 0:00:00


In [ ]:
# @title
# ============================================================
# IMPORTS
# ============================================================
import os
import gc
import warnings
import numpy as np
import pandas as pd
import librosa
import gudhi as gd

from tqdm import tqdm

from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest
from sklearn.mixture import GaussianMixture
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

In [ ]:
# @title
# ============================================================
# SETTINGS
# ============================================================

ROOT_FOLDER = "/content/test"   # folder containing 200 wav files

OUTPUT_FEATURES_XLSX = "/content/cubical_mel_tda_features.xlsx"

# Mel parameters
SR = None          # keep original sampling rate
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128

# Cubical persistence
EPS = 1e-12

In [ ]:
# @title
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def find_wav_files(root_folder):
    wav_files = []

    for root, _, files in os.walk(root_folder):
        for f in files:
            if f.lower().endswith(".wav"):
                wav_files.append(os.path.join(root, f))

    return sorted(wav_files)


def get_label_from_file_id(file_id):
    fid = str(file_id).lower()

    if "anomaly" in fid:
        return "anomaly"
    elif "normal" in fid:
        return "normal"
    else:
        return "unknown"


def load_wav_mono(path):
    audio, sample_rate = librosa.load(path, sr=SR, mono=True)

    if len(audio) == 0:
        raise ValueError("Empty audio signal")

    std = np.std(audio)

    if std < EPS:
        raise ValueError("Zero-std audio signal")

    audio = (audio - np.mean(audio)) / (std + EPS)

    return audio.astype(np.float64), sample_rate

In [ ]:
# @title
# ============================================================
# MEL SPECTROGRAM
# ============================================================

def compute_mel_spectrogram(audio, sample_rate):
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sample_rate,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        power=2.0
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    return mel_db

In [ ]:
# @title
# ============================================================
# CUBICAL COMPLEX PERSISTENCE
# ============================================================

def prepare_cubical_image(mel_db):
    """
    Gudhi CubicalComplex uses lower-star filtration.

    Since strong sound energy is important,
    we use -mel_db so high-energy regions appear earlier.
    """

    cubical_image = -mel_db

    cubical_image = (cubical_image - cubical_image.min()) / (
        cubical_image.max() - cubical_image.min() + EPS
    )

    return cubical_image.astype(np.float64)


def clean_diagram(diagram):
    diagram = np.asarray(diagram, dtype=np.float64)

    if diagram.size == 0:
        return np.empty((0, 2))

    if diagram.ndim != 2 or diagram.shape[1] != 2:
        return np.empty((0, 2))

    b = diagram[:, 0]
    d = diagram[:, 1]

    finite_deaths = d[np.isfinite(d)]
    dmax = np.max(finite_deaths) if len(finite_deaths) > 0 else 1.0

    d = np.where(np.isposinf(d), dmax, d)

    mask = np.isfinite(b) & np.isfinite(d) & (d > b)

    if not np.any(mask):
        return np.empty((0, 2))

    return np.column_stack([b[mask], d[mask]])


def compute_cubical_persistence_diagrams(mel_db):
    cubical_image = prepare_cubical_image(mel_db)

    cubical_complex = gd.CubicalComplex(
        top_dimensional_cells=cubical_image
    )

    cubical_complex.persistence()

    H0 = cubical_complex.persistence_intervals_in_dimension(0)
    H1 = cubical_complex.persistence_intervals_in_dimension(1)

    H0 = clean_diagram(H0)
    H1 = clean_diagram(H1)

    return H0, H1

In [ ]:
# @title
# ============================================================
# FULL TDA FEATURE EXTRACTION CODE
# ============================================================

import numpy as np

try:
    from scipy.stats import skew, kurtosis
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False


EPS = 1e-12


# ============================================================
# BASIC CLEANING
# ============================================================

def clean_diagram(diagram):
    """
    Cleans persistence diagram.

    Input:
        diagram: array-like of shape (n_points, 2)
                 columns = [birth, death]

    Output:
        cleaned diagram with finite birth/death and death > birth
    """

    if diagram is None:
        return np.empty((0, 2), dtype=np.float64)

    diagram = np.asarray(diagram, dtype=np.float64)

    if diagram.size == 0:
        return np.empty((0, 2), dtype=np.float64)

    if diagram.ndim != 2 or diagram.shape[1] < 2:
        return np.empty((0, 2), dtype=np.float64)

    diagram = diagram[:, :2]

    # Remove NaN rows
    diagram = diagram[~np.isnan(diagram).any(axis=1)]

    if len(diagram) == 0:
        return np.empty((0, 2), dtype=np.float64)

    # Remove infinite values
    finite_mask = np.isfinite(diagram[:, 0]) & np.isfinite(diagram[:, 1])
    diagram = diagram[finite_mask]

    if len(diagram) == 0:
        return np.empty((0, 2), dtype=np.float64)

    # Keep only valid persistence points
    diagram = diagram[diagram[:, 1] > diagram[:, 0]]

    return diagram.astype(np.float64)


def lifetimes(diagram):
    diagram = clean_diagram(diagram)

    if len(diagram) == 0:
        return np.empty(0, dtype=np.float64)

    return np.maximum(diagram[:, 1] - diagram[:, 0], 0.0)


# ============================================================
# SIMPLE STATISTICAL HELPERS
# ============================================================

def safe_mean(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.mean(x)) if len(x) > 0 else 0.0


def safe_std(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.std(x)) if len(x) > 0 else 0.0


def safe_var(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.var(x)) if len(x) > 0 else 0.0


def safe_min(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.min(x)) if len(x) > 0 else 0.0


def safe_max(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.max(x)) if len(x) > 0 else 0.0


def safe_median(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.median(x)) if len(x) > 0 else 0.0


def safe_quantile(x, q):
    x = np.asarray(x, dtype=np.float64)
    return float(np.quantile(x, q)) if len(x) > 0 else 0.0


def safe_iqr(x):
    x = np.asarray(x, dtype=np.float64)
    if len(x) == 0:
        return 0.0
    return float(np.quantile(x, 0.75) - np.quantile(x, 0.25))


def safe_range(x):
    x = np.asarray(x, dtype=np.float64)
    if len(x) == 0:
        return 0.0
    return float(np.max(x) - np.min(x))


def safe_skew(x):
    x = np.asarray(x, dtype=np.float64)

    if len(x) < 3:
        return 0.0

    if np.std(x) <= EPS:
        return 0.0

    if SCIPY_AVAILABLE:
        val = skew(x)
        return float(val) if np.isfinite(val) else 0.0

    # Manual fallback
    z = (x - np.mean(x)) / (np.std(x) + EPS)
    return float(np.mean(z ** 3))


def safe_kurtosis(x):
    x = np.asarray(x, dtype=np.float64)

    if len(x) < 4:
        return 0.0

    if np.std(x) <= EPS:
        return 0.0

    if SCIPY_AVAILABLE:
        val = kurtosis(x, fisher=True)
        return float(val) if np.isfinite(val) else 0.0

    # Manual fallback, Fisher kurtosis
    z = (x - np.mean(x)) / (np.std(x) + EPS)
    return float(np.mean(z ** 4) - 3.0)


# ============================================================
# ENTROPY / TAIL FEATURES
# ============================================================

def persistence_entropy_from_lifetimes(L):
    L = np.asarray(L, dtype=np.float64)

    if len(L) == 0:
        return 0.0

    total = np.sum(L)

    if total <= EPS:
        return 0.0

    p = L / (total + EPS)

    return float(-np.sum(p * np.log(p + EPS)))


def normalized_persistence_entropy(L):
    L = np.asarray(L, dtype=np.float64)

    if len(L) <= 1:
        return 0.0

    ent = persistence_entropy_from_lifetimes(L)

    return float(ent / (np.log(len(L)) + EPS))


def tail_share_from_lifetimes(L, q=0.90):
    L = np.asarray(L, dtype=np.float64)

    if len(L) == 0:
        return 0.0

    total = np.sum(L)

    if total <= EPS:
        return 0.0

    threshold = np.quantile(L, q)

    return float(np.sum(L[L >= threshold]) / (total + EPS))


def top_k_share(L, k=1):
    L = np.asarray(L, dtype=np.float64)

    if len(L) == 0:
        return 0.0

    total = np.sum(L)

    if total <= EPS:
        return 0.0

    L_sorted = np.sort(L)[::-1]
    k = min(k, len(L_sorted))

    return float(np.sum(L_sorted[:k]) / (total + EPS))


def vector_entropy(x):
    x = np.asarray(x, dtype=np.float64)
    x = np.abs(x)

    if len(x) == 0:
        return 0.0

    total = np.sum(x)

    if total <= EPS:
        return 0.0

    p = x / (total + EPS)

    return float(-np.sum(p * np.log(p + EPS)))


# ============================================================
# BETTI CURVE FEATURES
# ============================================================

def betti_curve(diagram, grid_size=100):
    diagram = clean_diagram(diagram)

    if len(diagram) == 0:
        return np.zeros(grid_size), np.linspace(0, 1, grid_size)

    births = diagram[:, 0]
    deaths = diagram[:, 1]

    t_min = np.min(births)
    t_max = np.max(deaths)

    if t_max <= t_min:
        return np.zeros(grid_size), np.linspace(0, 1, grid_size)

    grid = np.linspace(t_min, t_max, grid_size)

    beta = np.zeros(grid_size)

    for i, t in enumerate(grid):
        beta[i] = np.sum((births <= t) & (t < deaths))

    return beta, grid


def extract_betti_features(diagram, prefix, grid_size=100):
    beta, grid = betti_curve(diagram, grid_size=grid_size)

    feats = {}

    feats[f"{prefix}_betti_auc"] = float(np.trapz(beta, grid))
    feats[f"{prefix}_betti_mean"] = safe_mean(beta)
    feats[f"{prefix}_betti_std"] = safe_std(beta)
    feats[f"{prefix}_betti_var"] = safe_var(beta)
    feats[f"{prefix}_betti_max"] = safe_max(beta)
    feats[f"{prefix}_betti_min"] = safe_min(beta)
    feats[f"{prefix}_betti_l1"] = float(np.sum(np.abs(beta)))
    feats[f"{prefix}_betti_l2"] = float(np.sqrt(np.sum(beta ** 2)))
    feats[f"{prefix}_betti_energy"] = float(np.sum(beta ** 2))
    feats[f"{prefix}_betti_entropy"] = vector_entropy(beta)

    if len(beta) > 0:
        feats[f"{prefix}_betti_argmax"] = float(grid[int(np.argmax(beta))])
    else:
        feats[f"{prefix}_betti_argmax"] = 0.0

    # Number of local peaks
    if len(beta) >= 3:
        peaks = np.sum((beta[1:-1] > beta[:-2]) & (beta[1:-1] > beta[2:]))
        feats[f"{prefix}_betti_num_peaks"] = int(peaks)
    else:
        feats[f"{prefix}_betti_num_peaks"] = 0

    return feats


# ============================================================
# PERSISTENCE LANDSCAPE FEATURES
# ============================================================

def persistence_landscape_values(diagram, grid_size=100, num_layers=5):
    """
    Simple persistence landscape implementation.

    For each persistence point (b, d), triangle function:
        lambda(t) = max(0, min(t-b, d-t))

    Landscape layers are ordered triangle heights at each grid point.
    """

    diagram = clean_diagram(diagram)

    if len(diagram) == 0:
        grid = np.linspace(0, 1, grid_size)
        landscape = np.zeros((num_layers, grid_size))
        return landscape, grid

    births = diagram[:, 0]
    deaths = diagram[:, 1]

    t_min = np.min(births)
    t_max = np.max(deaths)

    if t_max <= t_min:
        grid = np.linspace(0, 1, grid_size)
        landscape = np.zeros((num_layers, grid_size))
        return landscape, grid

    grid = np.linspace(t_min, t_max, grid_size)

    triangle_values = []

    for b, d in diagram:
        values = np.maximum(0.0, np.minimum(grid - b, d - grid))
        triangle_values.append(values)

    triangle_values = np.asarray(triangle_values)

    landscape = np.zeros((num_layers, grid_size))

    for j in range(grid_size):
        vals = np.sort(triangle_values[:, j])[::-1]

        for k in range(min(num_layers, len(vals))):
            landscape[k, j] = vals[k]

    return landscape, grid


def extract_landscape_features(diagram, prefix, grid_size=100, num_layers=5):
    landscape, grid = persistence_landscape_values(
        diagram,
        grid_size=grid_size,
        num_layers=num_layers
    )

    flat = landscape.ravel()

    feats = {}

    feats[f"{prefix}_landscape_auc"] = float(np.sum([np.trapz(layer, grid) for layer in landscape]))
    feats[f"{prefix}_landscape_mean"] = safe_mean(flat)
    feats[f"{prefix}_landscape_std"] = safe_std(flat)
    feats[f"{prefix}_landscape_var"] = safe_var(flat)
    feats[f"{prefix}_landscape_max"] = safe_max(flat)
    feats[f"{prefix}_landscape_l1"] = float(np.sum(np.abs(flat)))
    feats[f"{prefix}_landscape_l2"] = float(np.sqrt(np.sum(flat ** 2)))
    feats[f"{prefix}_landscape_energy"] = float(np.sum(flat ** 2))
    feats[f"{prefix}_landscape_entropy"] = vector_entropy(flat)

    # Per-layer summaries
    for k in range(landscape.shape[0]):
        layer = landscape[k]
        feats[f"{prefix}_landscape_layer{k+1}_auc"] = float(np.trapz(layer, grid))
        feats[f"{prefix}_landscape_layer{k+1}_max"] = safe_max(layer)
        feats[f"{prefix}_landscape_layer{k+1}_mean"] = safe_mean(layer)

    return feats


# ============================================================
# PERSISTENCE SILHOUETTE FEATURES
# ============================================================

def persistence_silhouette_values(diagram, grid_size=100, power=1.0):
    """
    Persistence silhouette.

    Weighted average of triangle functions.
    Weight = lifetime^power
    """

    diagram = clean_diagram(diagram)

    if len(diagram) == 0:
        grid = np.linspace(0, 1, grid_size)
        silhouette = np.zeros(grid_size)
        return silhouette, grid

    births = diagram[:, 0]
    deaths = diagram[:, 1]
    L = deaths - births

    t_min = np.min(births)
    t_max = np.max(deaths)

    if t_max <= t_min:
        grid = np.linspace(0, 1, grid_size)
        silhouette = np.zeros(grid_size)
        return silhouette, grid

    grid = np.linspace(t_min, t_max, grid_size)

    weights = L ** power

    if np.sum(weights) <= EPS:
        silhouette = np.zeros(grid_size)
        return silhouette, grid

    silhouette = np.zeros(grid_size)

    for w, b, d in zip(weights, births, deaths):
        triangle = np.maximum(0.0, np.minimum(grid - b, d - grid))
        silhouette += w * triangle

    silhouette = silhouette / (np.sum(weights) + EPS)

    return silhouette, grid


def extract_silhouette_features(diagram, prefix, grid_size=100, power=1.0):
    silhouette, grid = persistence_silhouette_values(
        diagram,
        grid_size=grid_size,
        power=power
    )

    feats = {}

    feats[f"{prefix}_silhouette_auc"] = float(np.trapz(silhouette, grid))
    feats[f"{prefix}_silhouette_mean"] = safe_mean(silhouette)
    feats[f"{prefix}_silhouette_std"] = safe_std(silhouette)
    feats[f"{prefix}_silhouette_var"] = safe_var(silhouette)
    feats[f"{prefix}_silhouette_max"] = safe_max(silhouette)
    feats[f"{prefix}_silhouette_l1"] = float(np.sum(np.abs(silhouette)))
    feats[f"{prefix}_silhouette_l2"] = float(np.sqrt(np.sum(silhouette ** 2)))
    feats[f"{prefix}_silhouette_energy"] = float(np.sum(silhouette ** 2))
    feats[f"{prefix}_silhouette_entropy"] = vector_entropy(silhouette)

    return feats


# ============================================================
# PERSISTENCE IMAGE FEATURES
# ============================================================

def persistence_image(diagram, grid_size=20, sigma=0.1):
    """
    Simple persistence image.

    Coordinates:
        x = birth
        y = lifetime = death - birth

    Gaussian density placed at each point.
    """

    diagram = clean_diagram(diagram)

    if len(diagram) == 0:
        return np.zeros((grid_size, grid_size))

    births = diagram[:, 0]
    deaths = diagram[:, 1]
    pers = deaths - births

    if len(pers) == 0 or np.max(pers) <= EPS:
        return np.zeros((grid_size, grid_size))

    x_min, x_max = np.min(births), np.max(births)
    y_min, y_max = 0.0, np.max(pers)

    if x_max <= x_min:
        x_max = x_min + 1.0

    if y_max <= y_min:
        y_max = y_min + 1.0

    xs = np.linspace(x_min, x_max, grid_size)
    ys = np.linspace(y_min, y_max, grid_size)

    X, Y = np.meshgrid(xs, ys)

    img = np.zeros((grid_size, grid_size))

    for b, p in zip(births, pers):
        # Weight by persistence
        weight = p

        img += weight * np.exp(
            -((X - b) ** 2 + (Y - p) ** 2) / (2 * sigma ** 2 + EPS)
        )

    return img


def extract_persistence_image_features(diagram, prefix, grid_size=20, sigma=0.1):
    img = persistence_image(
        diagram,
        grid_size=grid_size,
        sigma=sigma
    )

    flat = img.ravel()

    feats = {}

    feats[f"{prefix}_pimage_sum"] = float(np.sum(flat))
    feats[f"{prefix}_pimage_mean"] = safe_mean(flat)
    feats[f"{prefix}_pimage_std"] = safe_std(flat)
    feats[f"{prefix}_pimage_var"] = safe_var(flat)
    feats[f"{prefix}_pimage_max"] = safe_max(flat)
    feats[f"{prefix}_pimage_min"] = safe_min(flat)
    feats[f"{prefix}_pimage_energy"] = float(np.sum(flat ** 2))
    feats[f"{prefix}_pimage_l1"] = float(np.sum(np.abs(flat)))
    feats[f"{prefix}_pimage_l2"] = float(np.sqrt(np.sum(flat ** 2)))
    feats[f"{prefix}_pimage_entropy"] = vector_entropy(flat)
    feats[f"{prefix}_pimage_nonzero_ratio"] = float(np.mean(flat > EPS))

    return feats


# ============================================================
# CARLSSON COORDINATES
# ============================================================

def extract_carlsson_coordinates(diagram, prefix):
    diagram = clean_diagram(diagram)

    feats = {}

    if len(diagram) == 0:
        feats[f"{prefix}_Carlsson_f1"] = 0.0
        feats[f"{prefix}_Carlsson_f2"] = 0.0
        feats[f"{prefix}_Carlsson_f3"] = 0.0
        feats[f"{prefix}_Carlsson_f4"] = 0.0
        return feats

    b = diagram[:, 0]
    d = diagram[:, 1]
    L = d - b

    feats[f"{prefix}_Carlsson_f1"] = float(np.sum(b * d))
    feats[f"{prefix}_Carlsson_f2"] = float(np.sum(L))
    feats[f"{prefix}_Carlsson_f3"] = float(np.sum((b ** 2) * (L ** 4)))
    feats[f"{prefix}_Carlsson_f4"] = float(np.sum((d ** 2) * (L ** 4)))

    return feats


# ============================================================
# WEIGHTED MOMENTS
# ============================================================

def weighted_mean(x, w):
    x = np.asarray(x, dtype=np.float64)
    w = np.asarray(w, dtype=np.float64)

    if len(x) == 0 or np.sum(w) <= EPS:
        return 0.0

    return float(np.sum(w * x) / (np.sum(w) + EPS))


def weighted_std(x, w):
    x = np.asarray(x, dtype=np.float64)
    w = np.asarray(w, dtype=np.float64)

    if len(x) == 0 or np.sum(w) <= EPS:
        return 0.0

    mu = weighted_mean(x, w)

    return float(np.sqrt(np.sum(w * (x - mu) ** 2) / (np.sum(w) + EPS)))


def extract_weighted_moment_features(diagram, prefix):
    diagram = clean_diagram(diagram)

    feats = {}

    if len(diagram) == 0:
        feats[f"{prefix}_weighted_birth_mean"] = 0.0
        feats[f"{prefix}_weighted_birth_std"] = 0.0
        feats[f"{prefix}_weighted_death_mean"] = 0.0
        feats[f"{prefix}_weighted_death_std"] = 0.0
        feats[f"{prefix}_weighted_midlife_mean"] = 0.0
        feats[f"{prefix}_weighted_midlife_std"] = 0.0
        return feats

    b = diagram[:, 0]
    d = diagram[:, 1]
    L = d - b
    midlife = (b + d) / 2.0

    feats[f"{prefix}_weighted_birth_mean"] = weighted_mean(b, L)
    feats[f"{prefix}_weighted_birth_std"] = weighted_std(b, L)

    feats[f"{prefix}_weighted_death_mean"] = weighted_mean(d, L)
    feats[f"{prefix}_weighted_death_std"] = weighted_std(d, L)

    feats[f"{prefix}_weighted_midlife_mean"] = weighted_mean(midlife, L)
    feats[f"{prefix}_weighted_midlife_std"] = weighted_std(midlife, L)

    return feats


# ============================================================
# MAIN PER-DIMENSION FEATURE EXTRACTOR
# ============================================================

def extract_diagram_features(
    diagram,
    prefix,
    grid_size=100,
    landscape_layers=5,
    pimage_grid_size=20,
    pimage_sigma=0.1
):
    diagram = clean_diagram(diagram)

    feats = {}

    if len(diagram) == 0:
        b = np.empty(0)
        d = np.empty(0)
        L = np.empty(0)
    else:
        b = diagram[:, 0]
        d = diagram[:, 1]
        L = d - b

    midlife = (b + d) / 2.0 if len(diagram) > 0 else np.empty(0)
    diag_dist = L / np.sqrt(2.0) if len(L) > 0 else np.empty(0)

    birth_death_ratio = b / (d + EPS) if len(diagram) > 0 else np.empty(0)

    # --------------------------------------------------------
    # Lifetime statistics
    # --------------------------------------------------------
    feats[f"{prefix}_num_points"] = int(len(L))

    feats[f"{prefix}_sum_lifetime"] = float(np.sum(L)) if len(L) > 0 else 0.0
    feats[f"{prefix}_mean_lifetime"] = safe_mean(L)
    feats[f"{prefix}_std_lifetime"] = safe_std(L)
    feats[f"{prefix}_var_lifetime"] = safe_var(L)
    feats[f"{prefix}_min_lifetime"] = safe_min(L)
    feats[f"{prefix}_max_lifetime"] = safe_max(L)
    feats[f"{prefix}_median_lifetime"] = safe_median(L)

    feats[f"{prefix}_q10_lifetime"] = safe_quantile(L, 0.10)
    feats[f"{prefix}_q25_lifetime"] = safe_quantile(L, 0.25)
    feats[f"{prefix}_q75_lifetime"] = safe_quantile(L, 0.75)
    feats[f"{prefix}_q90_lifetime"] = safe_quantile(L, 0.90)
    feats[f"{prefix}_q95_lifetime"] = safe_quantile(L, 0.95)

    feats[f"{prefix}_iqr_lifetime"] = safe_iqr(L)
    feats[f"{prefix}_range_lifetime"] = safe_range(L)

    feats[f"{prefix}_lifetime_skew"] = safe_skew(L)
    feats[f"{prefix}_lifetime_kurtosis"] = safe_kurtosis(L)

    # --------------------------------------------------------
    # Entropy
    # --------------------------------------------------------
    feats[f"{prefix}_persistence_entropy"] = persistence_entropy_from_lifetimes(L)
    feats[f"{prefix}_normalized_persistence_entropy"] = normalized_persistence_entropy(L)

    if np.sum(L) > EPS:
        feats[f"{prefix}_entropy_lifetime_ratio"] = (
            feats[f"{prefix}_persistence_entropy"] / (np.sum(L) + EPS)
        )
    else:
        feats[f"{prefix}_entropy_lifetime_ratio"] = 0.0

    # --------------------------------------------------------
    # Norms and energy
    # --------------------------------------------------------
    feats[f"{prefix}_l1_norm_lifetime"] = float(np.sum(np.abs(L)))
    feats[f"{prefix}_l2_norm_lifetime"] = float(np.sqrt(np.sum(L ** 2)))
    feats[f"{prefix}_linf_norm_lifetime"] = safe_max(np.abs(L))
    feats[f"{prefix}_lifetime_energy"] = float(np.sum(L ** 2))
    feats[f"{prefix}_lifetime_rms"] = float(np.sqrt(np.mean(L ** 2))) if len(L) > 0 else 0.0

    # --------------------------------------------------------
    # Tail shares
    # --------------------------------------------------------
    feats[f"{prefix}_tail_share_q80"] = tail_share_from_lifetimes(L, 0.80)
    feats[f"{prefix}_tail_share_q90"] = tail_share_from_lifetimes(L, 0.90)
    feats[f"{prefix}_tail_share_q95"] = tail_share_from_lifetimes(L, 0.95)

    feats[f"{prefix}_top1_share"] = top_k_share(L, 1)
    feats[f"{prefix}_top3_share"] = top_k_share(L, 3)
    feats[f"{prefix}_top5_share"] = top_k_share(L, 5)

    # --------------------------------------------------------
    # Birth statistics
    # --------------------------------------------------------
    feats[f"{prefix}_birth_mean"] = safe_mean(b)
    feats[f"{prefix}_birth_std"] = safe_std(b)
    feats[f"{prefix}_birth_var"] = safe_var(b)
    feats[f"{prefix}_birth_min"] = safe_min(b)
    feats[f"{prefix}_birth_max"] = safe_max(b)
    feats[f"{prefix}_birth_median"] = safe_median(b)
    feats[f"{prefix}_birth_q25"] = safe_quantile(b, 0.25)
    feats[f"{prefix}_birth_q75"] = safe_quantile(b, 0.75)
    feats[f"{prefix}_birth_iqr"] = safe_iqr(b)
    feats[f"{prefix}_birth_skew"] = safe_skew(b)
    feats[f"{prefix}_birth_kurtosis"] = safe_kurtosis(b)

    # --------------------------------------------------------
    # Death statistics
    # --------------------------------------------------------
    feats[f"{prefix}_death_mean"] = safe_mean(d)
    feats[f"{prefix}_death_std"] = safe_std(d)
    feats[f"{prefix}_death_var"] = safe_var(d)
    feats[f"{prefix}_death_min"] = safe_min(d)
    feats[f"{prefix}_death_max"] = safe_max(d)
    feats[f"{prefix}_death_median"] = safe_median(d)
    feats[f"{prefix}_death_q25"] = safe_quantile(d, 0.25)
    feats[f"{prefix}_death_q75"] = safe_quantile(d, 0.75)
    feats[f"{prefix}_death_iqr"] = safe_iqr(d)
    feats[f"{prefix}_death_skew"] = safe_skew(d)
    feats[f"{prefix}_death_kurtosis"] = safe_kurtosis(d)

    # --------------------------------------------------------
    # Geometry features
    # --------------------------------------------------------
    feats[f"{prefix}_mean_birth_death_ratio"] = safe_mean(birth_death_ratio)
    feats[f"{prefix}_std_birth_death_ratio"] = safe_std(birth_death_ratio)

    feats[f"{prefix}_mean_midlife"] = safe_mean(midlife)
    feats[f"{prefix}_std_midlife"] = safe_std(midlife)
    feats[f"{prefix}_max_midlife"] = safe_max(midlife)
    feats[f"{prefix}_min_midlife"] = safe_min(midlife)

    feats[f"{prefix}_mean_diag_distance"] = safe_mean(diag_dist)
    feats[f"{prefix}_sum_diag_distance"] = float(np.sum(diag_dist)) if len(diag_dist) > 0 else 0.0
    feats[f"{prefix}_max_diag_distance"] = safe_max(diag_dist)
    feats[f"{prefix}_std_diag_distance"] = safe_std(diag_dist)

    # --------------------------------------------------------
    # Betti curve features
    # --------------------------------------------------------
    feats.update(
        extract_betti_features(
            diagram,
            prefix,
            grid_size=grid_size
        )
    )

    # --------------------------------------------------------
    # Persistence landscape features
    # --------------------------------------------------------
    feats.update(
        extract_landscape_features(
            diagram,
            prefix,
            grid_size=grid_size,
            num_layers=landscape_layers
        )
    )

    # --------------------------------------------------------
    # Persistence silhouette features
    # --------------------------------------------------------
    feats.update(
        extract_silhouette_features(
            diagram,
            prefix,
            grid_size=grid_size,
            power=1.0
        )
    )

    # --------------------------------------------------------
    # Persistence image features
    # --------------------------------------------------------
    feats.update(
        extract_persistence_image_features(
            diagram,
            prefix,
            grid_size=pimage_grid_size,
            sigma=pimage_sigma
        )
    )

    # --------------------------------------------------------
    # Carlsson coordinates
    # --------------------------------------------------------
    feats.update(
        extract_carlsson_coordinates(
            diagram,
            prefix
        )
    )

    # --------------------------------------------------------
    # Weighted moment features
    # --------------------------------------------------------
    feats.update(
        extract_weighted_moment_features(
            diagram,
            prefix
        )
    )

    return feats


# ============================================================
# H0 / H1 INTERACTION FEATURES
# ============================================================

def safe_ratio(a, b):
    if abs(b) <= EPS:
        return 0.0
    return float(a / (b + EPS))


def extract_h0_h1_interaction_features(feats):
    interaction = {}

    interaction["H1_to_H0_sum_lifetime_ratio"] = safe_ratio(
        feats.get("H1_sum_lifetime", 0.0),
        feats.get("H0_sum_lifetime", 0.0)
    )

    interaction["H1_to_H0_max_lifetime_ratio"] = safe_ratio(
        feats.get("H1_max_lifetime", 0.0),
        feats.get("H0_max_lifetime", 0.0)
    )

    interaction["H1_to_H0_num_points_ratio"] = safe_ratio(
        feats.get("H1_num_points", 0.0),
        feats.get("H0_num_points", 0.0)
    )

    interaction["H1_to_H0_entropy_ratio"] = safe_ratio(
        feats.get("H1_persistence_entropy", 0.0),
        feats.get("H0_persistence_entropy", 0.0)
    )

    interaction["H1_to_H0_energy_ratio"] = safe_ratio(
        feats.get("H1_lifetime_energy", 0.0),
        feats.get("H0_lifetime_energy", 0.0)
    )

    interaction["H1_minus_H0_sum_lifetime"] = float(
        feats.get("H1_sum_lifetime", 0.0) -
        feats.get("H0_sum_lifetime", 0.0)
    )

    interaction["H1_minus_H0_max_lifetime"] = float(
        feats.get("H1_max_lifetime", 0.0) -
        feats.get("H0_max_lifetime", 0.0)
    )

    interaction["H1_minus_H0_entropy"] = float(
        feats.get("H1_persistence_entropy", 0.0) -
        feats.get("H0_persistence_entropy", 0.0)
    )

    return interaction


# ============================================================
# FINAL FULL FEATURE FUNCTION
# ============================================================

def extract_full_tda_features(
    H0,
    H1,
    grid_size=100,
    landscape_layers=5,
    pimage_grid_size=20,
    pimage_sigma=0.1
):
    """
    Full feature extractor from H0 and H1 persistence diagrams.

    Parameters
    ----------
    H0 : array-like
        H0 persistence diagram, shape (n_points, 2)

    H1 : array-like
        H1 persistence diagram, shape (n_points, 2)

    Returns
    -------
    feats : dict
        Dictionary of all extracted TDA features
    """

    feats = {}

    feats.update(
        extract_diagram_features(
            H0,
            prefix="H0",
            grid_size=grid_size,
            landscape_layers=landscape_layers,
            pimage_grid_size=pimage_grid_size,
            pimage_sigma=pimage_sigma
        )
    )

    feats.update(
        extract_diagram_features(
            H1,
            prefix="H1",
            grid_size=grid_size,
            landscape_layers=landscape_layers,
            pimage_grid_size=pimage_grid_size,
            pimage_sigma=pimage_sigma
        )
    )

    feats.update(
        extract_h0_h1_interaction_features(feats)
    )

    return feats




In [ ]:
# @title
# ============================================================
# SINGLE WAV -> FULL FEATURE ROW
# ============================================================

def extract_cubical_mel_tda_features_for_wav(path):
    """
    Extracts full cubical-persistence TDA features from one WAV file.

    Pipeline:
        WAV
        -> mono audio
        -> Mel spectrogram
        -> cubical persistence diagrams H0, H1
        -> full TDA feature extraction
        -> one feature row
    """

    # ----------------------------
    # Load audio
    # ----------------------------
    audio, sample_rate = load_wav_mono(path)

    file_id = os.path.basename(path)
    label = get_label_from_file_id(file_id)

    # ----------------------------
    # Mel spectrogram
    # ----------------------------
    mel_db = compute_mel_spectrogram(audio, sample_rate)

    # ----------------------------
    # Cubical persistence
    # ----------------------------
    H0, H1 = compute_cubical_persistence_diagrams(mel_db)

    # ----------------------------
    # Full TDA feature extraction
    # ----------------------------
    feats = extract_full_tda_features(
        H0,
        H1,
        grid_size=100,
        landscape_layers=5,
        pimage_grid_size=20,
        pimage_sigma=0.1
    )

    # ----------------------------
    # Metadata row
    # ----------------------------
    row = {
        "file_id": file_id,
        "file_path": path,
        "label": label,
        "duration_sec": len(audio) / sample_rate,
        "sample_rate": sample_rate,
        "n_samples": len(audio),
        "n_mel_bins": mel_db.shape[0],
        "n_mel_frames": mel_db.shape[1],
        "H0_points_raw": len(H0),
        "H1_points_raw": len(H1),
    }

    row.update(feats)

    return row


# ============================================================
# FEATURE EXTRACTION FOR ALL WAV FILES
# ============================================================

wav_files = find_wav_files(ROOT_FOLDER)

print("Number of WAV files found:", len(wav_files))

rows = []
failed = []

for path in tqdm(wav_files):
    try:
        row = extract_cubical_mel_tda_features_for_wav(path)
        rows.append(row)

    except Exception as e:
        failed.append({
            "file_id": os.path.basename(path),
            "file_path": path,
            "error": str(e)
        })

    gc.collect()


# ============================================================
# CREATE FEATURE DATAFRAME
# ============================================================

if len(rows) == 0:
    raise ValueError("No valid WAV files processed.")

features_df = pd.DataFrame(rows)

# Clean infinite and missing values
features_df = features_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Save features
features_df.to_excel(OUTPUT_FEATURES_XLSX, index=False)

print("Feature extraction completed.")
print("Feature table shape:", features_df.shape)
print("Saved features to:", OUTPUT_FEATURES_XLSX)

display(features_df.head())


# ============================================================
# SAVE FAILED FILES, IF ANY
# ============================================================

if len(failed) > 0:
    failed_df = pd.DataFrame(failed)
    failed_path = "/content/failed_cubical_files.xlsx"
    failed_df.to_excel(failed_path, index=False)

    print("Failed files:", len(failed))
    print("Saved failed files to:", failed_path)
else:
    print("No failed files.")

Number of WAV files found: 1000


100%|██████████| 1000/1000 [07:40<00:00,  2.17it/s]


Feature extraction completed.
Feature table shape: (1000, 276)
Saved features to: /content/cubical_mel_tda_features.xlsx


,file_id,file_path,label,duration_sec,sample_rate,n_samples,n_mel_bins,n_mel_frames,H0_points_raw,H1_points_raw,...,H1_weighted_midlife_mean,H1_weighted_midlife_std,H1_to_H0_sum_lifetime_ratio,H1_to_H0_max_lifetime_ratio,H1_to_H0_num_points_ratio,H1_to_H0_entropy_ratio,H1_to_H0_energy_ratio,H1_minus_H0_sum_lifetime,H1_minus_H0_max_lifetime,H1_minus_H0_entropy
0,section_00_source_train_normal_0000_id_A2_spd_...,/content/test/section_00_source_train_normal_0...,normal,7.0,16000,112000,128,219,1878,2502,...,0.441213,0.119523,1.120220,0.969784,1.332268,1.035143,1.024459,8.590307,-0.008632,0.249519
1,section_00_source_train_normal_0001_id_A1_spd_...,/content/test/section_00_source_train_normal_0...,normal,7.0,16000,112000,128,219,1832,2436,...,0.475373,0.120408,1.128955,0.899139,1.329694,1.031871,1.077641,9.097903,-0.029247,0.226038
2,section_00_source_train_normal_0002_id_A1_spd_...,/content/test/section_00_source_train_normal_0...,normal,7.0,16000,112000,128,219,1855,2613,...,0.441080,0.117237,1.196675,1.160667,1.408625,1.038309,1.150818,13.433176,0.037592,0.273132
3,section_00_source_train_normal_0003_id_A1_spd_...,/content/test/section_00_source_train_normal_0...,normal,7.0,16000,112000,128,219,1769,2492,...,0.485344,0.114979,1.147076,0.962608,1.408705,1.045220,0.981125,9.543397,-0.008712,0.318167
4,section_00_source_train_normal_0004_id_A1_spd_...,/content/test/section_00_source_train_normal_0...,normal,7.0,16000,112000,128,219,1835,2468,...,0.457750,0.115897,1.109168,0.893769,1.344959,1.032949,1.006009,7.754535,-0.029610,0.234097


No failed files.
